# 找離群.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io


In [ ]:
print("請上傳包含多個 CSV 檔案的資料夾：")
uploaded_files = files.upload()


In [ ]:
category_1_angles = []
file_names = []


In [ ]:
def calculate_angle_change(x1, y1, x2, y2, x3, y3):

    a_x = x2 - x1
    a_y = y2 - y1

    b_x = x3 - x2
    b_y = y3 - y2

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = np.hypot(a_x, a_y)
    magnitude_b = np.hypot(b_x, b_y)
    if magnitude_a * magnitude_b == 0:
        return 0

    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))

    return np.arccos(cos_angle) * (180 / np.pi)


In [ ]:
for file_name in uploaded_files.keys():
    df = pd.read_csv(io.BytesIO(uploaded_files[file_name]))

    if 'kpts' in df.columns:
        kpts_columns = df['kpts'].str.split(',', expand=True)
        kpts_columns.columns = [f'kpt_{i+1}' for i in range(kpts_columns.shape[1])]
        df = df.drop(columns=[col for col in df.columns if col.startswith('kpt_')], errors='ignore')
        df = df.join(kpts_columns)

    required_kpts = ['kpt_31', 'kpt_32', 'kpt_34', 'kpt_35', 'kpt_40', 'kpt_41', 'kpt_44', 'kpt_45']
    if not all(kpt in df.columns for kpt in required_kpts):
        print(f"文件 {file_name} 缺少必要的關鍵點數據，請檢查文件內容。")
        continue

    x11 = df['kpt_31'].astype(float)
    y11 = df['kpt_32'].astype(float)
    x12 = df['kpt_34'].astype(float)
    y12 = df['kpt_35'].astype(float)
    x14 = df['kpt_40'].astype(float)
    y14 = df['kpt_41'].astype(float)
    x15 = df['kpt_44'].astype(float)
    y15 = df['kpt_45'].astype(float)
    x18 = (x11 + x12) / 2
    y18 = (y11 + y12) / 2

    angles = []
    for i in range(len(x18)):
        angle_change_value = calculate_angle_change(x14.iloc[i], y14.iloc[i], x18.iloc[i], y18.iloc[i], x15.iloc[i], y15.iloc[i])
        angles.append(angle_change_value)

    category_1_angles.append(angles)
    file_names.append(file_name)


In [ ]:
flat_angles = [angle for angles in category_1_angles for angle in angles]
Q1 = pd.Series(flat_angles).quantile(0.25)
Q3 = pd.Series(flat_angles).quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


In [ ]:
outliers_dict = {}
file_outliers_count = {}


In [ ]:
total_outliers = 0
total_values = 0
for i, angles in enumerate(category_1_angles):
    outliers = [(j, angle) for j, angle in enumerate(angles) if angle < lower_bound or angle > upper_bound]
    if outliers:
        outliers_dict[file_names[i]] = outliers
    total_outliers += len(outliers)
    total_values += len(angles)

    file_outliers_count[file_names[i]] = len(outliers)


In [ ]:
for file_name, outliers in outliers_dict.items():
    print(f"\n檔案: {file_name}")
    for row_idx, outlier_value in outliers:
        print(f"離群值: {outlier_value}，行數: {row_idx + 1}")
        print(f"該行的數據: \n{pd.read_csv(io.BytesIO(uploaded_files[file_name])).iloc[row_idx]}")


In [ ]:
print(f"\n總離群值數量: {total_outliers}")
print(f"總數據點數量: {total_values}")


In [ ]:
print("\n每個檔案的離群值數量：")
for file_name, count in file_outliers_count.items():
    print(f"檔案: {file_name}，離群值數量: {count}")
